# **Pathly**
## **M4 Design**
**Team Members**: Sammi Chong, Ellie Lu, Naseeha Masub, Saba Sohail @ Stony Brook University

---
### **Overview**
Pathly is a mobile public transportation app built with React Native and Expo for riders in the New York City and Long Island area. Riders can see nearby transit with live arrivals, search for a place, stop or line, compare real transit itineraries, and follow an active trip step by step. Since M3, riders can also sign in with Google, save lines and get alerts for them, see live problems with a planned trip, and use Pathly in a phone browser at https://pathly.web.app.

The app communicates with a Firebase backend:
- **Cloud Functions** do all of the transit work: reading each agency's GTFS schedule, merging GTFS-Realtime feeds, planning trips through the Google Routes API, estimating fares, computing walking paths, and sending service alerts. 
- **Cloud Firestore** stores rider data (profile, recent searches, previous destinations, saved places, and saved lines). 
- **Cloud Storage** holds versioned GTFS snapshots that a scheduled function refreshes every day.

This document updates the M3 design to match what we built, and calls out where the design changed and why.

---
### **What Changed Since M3**

| Area | M3 design | M4 design | Why |
|---|---|---|---|
| **Authentication** | Email and password | Email and password, plus Google sign-in | Google sign-in was the follow-up left open in M3. It works on the web today; on phones it needs a development build. |
| **Saved routes** | Saved places only | Saved lines, with favorites and per-line alert settings | Riders usually care about the same few lines |
| **Notifications** | Not started | Push alerts for saved lines, sent every 5 minutes, plus a "hurry up" nudge during a trip | Riders shouldn't have to open the app to learn about a disruption |
| **Trip results** | Google itineraries with Pathly fares | The same, plus a live check of each trip for cancellations, skipped stops, and tight connections | Google's plans can miss live changes |
| **Line browsing** | Stop and place search | Line search, an app-wide line index, and home map layers for every line of a mode | Riders also look for a line by name, not only a place |
| **Accessibility data** | High-contrast mode | Also shows which stops a wheelchair user can board at | Step-free access is part of planning a trip |
| **Offline use** | None | The MTA's official maps can be saved to the device | Riders lose signal underground |
| **Personalization** | None | Bean avatars and a custom destination pin | Makes the app feel like the rider's own |
| **Rider feedback** | None | Feedback survey and donation links | Lets riders report problems and support the project |
| **Platform** | Expo 57, React Native 0.86 | Expo 58, React Native 0.88 | Expo Go on our phones moved to SDK 58 |
| **Mobile web** | Tested on desktop browsers | Hosted on Firebase Hosting, with layout and scrolling fixes for iOS Safari | Testing on a real iPhone showed problems desktop browsers don't |
| **Maps in Expo Go** | Google Maps only | Apple Maps in Expo Go on iOS, Google Maps everywhere else | Expo Go on iOS has no Google Maps SDK, so the map was blank |
| **Testing** | About 500 automated tests | 612 automated tests, plus a manual test plan | New features added tests, and manual testing caught bugs they missed |


---
### **Architecture**

The architecture is unchanged from M3. The mobile app never calls Google or transit feeds directly with a secret key; it calls Firebase callable functions, which do that work and return small, ready-to-display results.

**How a request flows (example: Nearby):**
1. The app reads the rider's location with `expo-location`.
2. It calls the `findNearbyTransit` callable function, signed in through Firebase Authentication.
3. The function looks up nearby stops in the current GTFS snapshot's prebuilt index, then the app asks `getRoutesLiveStatus` for live arrivals for the routes it shows.
4. The function fetches and merges each agency's GTFS-Realtime feed, matches live trips to the schedule, and adds service alerts.
5. The app refreshes live data every 15 seconds while the screen is open, and again when the rider moves.

**Cloud Functions:**

| Function | Purpose |
|---|---|
| `findNearbyTransit` | Stops and routes near the rider |
| `getRouteLiveStatus` / `getRoutesLiveStatus` | Live arrivals, vehicle positions, delays and alerts for one or many routes |
| `getStopDepartures` | Upcoming departures at a stop, live where available, otherwise scheduled |
| `getNearestRouteStop` | The closest stop on a chosen route |
| `getRouteTransfers` | Lines a rider can transfer to along a route |
| `getRouteGeometry` | The shape of a route for drawing on the map |
| `getWalkingPath` | A walking path to a stop, from prebuilt walking tiles |
| `getAgencyLines` | An agency's lines, for the home map's line layers |
| `getAccessibleStops` | Which of an agency's stops a wheelchair user can board at |
| `planTrip` | Transit itineraries from the Google Routes API, plus fares, stop lists, and a live check of each trip |
| `searchTransitStops` | Stop search by name |
| `searchTransitRoutes` | Line search by name or number |
| `listTransitLines` | The line index the app keeps on the device |
| `describeTransitRoute` | Opens a line from a stop or transfer badge |
| `geocodeAddress` | Turns an address into coordinates |
| `reverseGeocode` | Names a point chosen on the map |
| `getBackendStatus` | Health check used by the admin tools |
| `refreshGtfsStaticData` | Scheduled daily at 04:00 New York time to refresh every agency's GTFS feed |
| `sendRouteAlerts` | Scheduled every 5 minutes to push new service alerts for riders' saved lines |

---
### **Stack**

The stack is the same as M3, with Expo and React Native upgraded and a few additions for notifications, Google sign-in, and web hosting. Firebase's authentication, database, storage, scheduled jobs and server code all live in one place, which lets us spend our time on transit features instead of infrastructure.

| Layer | Technology |
|---|---|
| **frontend** | React Native 0.88 + Expo 58, TypeScript |
| **maps** | `react-native-maps` on iOS/Android (Apple Maps in Expo Go on iOS), `@react-google-maps/api` on web |
| **place search** | Google Places Autocomplete (Places API (New)) |
| **backend** | Firebase (Blaze plan) |
| **server-side logic** | Cloud Functions for Firebase (Node 24, TypeScript) |
| **database** | Cloud Firestore |
| **file storage** | Firebase Cloud Storage (GTFS snapshots) |
| **authentication** | Firebase Authentication (email and password, Google sign-in) |
| **notifications** | `expo-notifications` and Expo's push service |
| **web hosting** | Firebase Hosting |
| **secrets / abuse protection** | Firebase Secret Manager, App Check, per-user rate limits |
| **trip planning** | Google Routes API |
| **CI** | GitHub Actions |

The repository is an npm workspace with two packages: `mobile/` (the Expo app) and `firebase/functions/` (the backend).

---
### **Design Goals**

- **Accuracy** - Live GTFS-Realtime data is merged with each agency's schedule, so riders see real arrivals instead of timetable guesses. Where an agency has no live feed (the Staten Island Ferry), departures come from the schedule and the app says so.
- **Simplicity** - The home screen opens straight to the map and nearby transit. Saved places sit in a strip on the home screen and as pins on the map, so common trips take one or two taps.
- **Scalability** - Every agency is described by one config entry in `gtfsAgencies.ts` (where its feeds live, how its live trips match the schedule, which secret it needs). Adding the Staten Island Ferry and the six MTA bus feeds only required new entries, not changes to the trip logic.
- **Speed** - Prebuilt indexes, cached snapshots, batched live-status requests and a device cache keep cold starts and repeat views fast.
- **Consistency** *(new)* - The app should behave the same on every phone. Screens now share one source for their size, so layouts stay in place on iOS Safari.

---
### **Design Specifications**

Status of the core features from M2 and M3:

| Feature | Implementation | Status |
|---|---|---|
| display nearby transit | device location + `findNearbyTransit` + walking-distance sorting | Done |
| display transit routes | route detail view with stops, map shape, transfers and live vehicles | Done |
| search for destination | Google Places Autocomplete + transit stop and line search | Done |
| compare routes | route results based on time, transfers and fare | Done |
| route navigation | en-route mode driven by location, resumable after restart, with a notice when the rider is off route | Done (rerouting and background navigation are future work) |
| view scheduled transit | GTFS static data, refreshed daily | Done |
| view real-time transit | GTFS-RT through MTA, MTA Bus Time, Swiftly and NYC Ferry feeds | Done |
| view service alerts | GTFS-RT service alerts shown as route notices | Done |
| login | Firebase Authentication (email and password, Google sign-in) | Done (Google sign-in on phones needs a development build) |
| user profiles | Cloud Firestore; includes bean avatars, data export, and account deletion | Done |
| save favorite routes/stops/locations | saved places and saved lines in Firestore; places show as home map pins | Done |
| settings | appearance, accessibility (high contrast), preferences | Done |
| admin / debug tools | mock time and location, request log, backend status | Done |
| notifications | Expo push + `sendRouteAlerts`; local "hurry up" nudge | Done |
| accessible stops | `getAccessibleStops` | Done |
| line layers | `getAgencyLines` + home map layers | Done |
| offline maps | MTA map PDFs saved on the device | Done |
| mobile web | Firebase Hosting + iOS Safari fixes | Done (other phone browsers not tested yet) |
| friends | Firebase | Not started |
| widgets | native iOS/Android + backend | Not started |

---
### **Frontend Design**

We kept dark blue as the primary color and Nunito as the font (Regular, SemiBold, Bold, ExtraBold and Black weights). The palette is unchanged from M3: light, dark, and high-contrast versions.

| Role | Light | Dark | Purpose |
|---|---|---|---|
| **primary** | #0B4F9C | #5B9BF8 | main buttons, active states, logo |
| **accent** | #A7D8FF | #2F5E93 | secondary buttons, highlights, selected elements |
| **background** | #F7FAFF | #000000 | main app background |
| **success** | #16A34A | #34C06B | confirmations, live transit |
| **warning** | #F97316 | #FB923C | delays, alerts, warnings |

High-contrast mode (Settings → Accessibility) darkens text and borders and deepens the primary color, for example primary #063A75 in light mode.

**Route colors:** each line uses its agency's official color (for example, subway lines use MTA bullet colors), and lines without an official color get one standard color per agency instead of a random one. The same rule runs on the backend and in the app so badges and map lines always match.

**Pathly's character:** the Pathly mascot appears on loading and empty states, riders can build a bean avatar for their profile, and they can customize the pin shown at their destination. Agency logos appear on route and trip screens.

**Main screens:**
- **Home** - the map with nearby transit in a single scrolling sheet, line layers, saved places and lines, recent trips, and a card to resume an active trip
- **Search** - places, transit stops and lines, with recent searches, and a way to choose a point on the map
- **Route Results** - itineraries to compare, with live trip checks, a leave-time picker, and mode and sort filters
- **Planned Trip Detail** - each leg of the trip on a map
- **En-route Navigation** - the current instruction, next stop, stops remaining and arrival estimate
- **Route Detail / Departures** - a line's stops, live vehicles, transfers and alerts, a direction switch, and alert settings for saved lines
- **Profile and Settings** - account, avatar, appearance, accessibility, offline maps, feedback and support links, and admin tools for admin accounts

The map has separate native and web versions (`*.native.tsx` / `*.web.tsx`) because the two platforms need different map libraries.

**Mobile web:** a custom `public/index.html` keeps the page itself from scrolling on iOS Safari, so only the app's own scroll areas move. `useViewportSize` gives every screen the same size to work from, and `useTouchScroll` follows the finger on iOS browsers for lists that sit over a draggable map.

---
### **Backend/APIs**

| Source | Used for |
|---|---|
| Google Maps (native SDKs and JavaScript API) | the map |
| Google Places API (New) | destination autocomplete |
| Google Routes API | transit itineraries (`planTrip`) |
| Google Geocoding | `geocodeAddress` |
| MTA GTFS and GTFS-RT | subway, LIRR, service alerts |
| MTA Bus Time | live MTA bus data (requires an API key) |
| Swiftly | live NICE and Suffolk County Transit data (requires an API key) |
| NYC Ferry | schedule and live trip updates |
| Staten Island Ferry | schedule only (no live feed) |
| Expo push service | delivers service alerts to riders' phones |

**Transit data pipeline.** Every day at 04:00 New York time, `refreshGtfsStaticData` downloads each agency's GTFS feed, validates it, and uploads it to Cloud Storage as a new version under `gtfs/v1/<agency>/`. Only after a feed passes validation does the job update that agency's `current.json` to point to it, so a broken or expired feed never replaces a good one. The same job builds small per-route index files for each version. Functions check for a new version at most every five minutes, keep the last good copy on that server instance, and fall back to the GTFS files bundled with the deployment if Storage is unavailable. Old versions are cleaned up once two newer ones exist.

**Service alerts.** Every 5 minutes, `sendRouteAlerts` finds saved lines with alerts turned on, reads each line's current alerts, and pushes new ones to the rider's devices. Riders can get alerts all the time or only during custom hours, or pause them. An alert already sent to a rider isn't sent again for 3 days, even if it stays active.

**Trip planning.** `planTrip` sends the origin, destination and time to the Google Routes API in transit mode, then adds Pathly's own fare estimates, stop lists and live data. A live check then compares each trip with the agencies' feeds to flag cancelled trips, skipped stops, and connections that are tight or already missed. Results are cached for 2 minutes and each rider can plan up to 20 trips per minute, which keeps us inside the free tier.

---
### **Database**

| Path | Contents |
|---|---|
| `users/{uid}` | profile: email, display name, avatar, push tokens, and whether alerts are paused |
| `users/{uid}/searches/{placeKey}` | a recent search: place and date searched |
| `users/{uid}/prevDests/{placeKey}` | a previous destination: place and date navigated |
| `users/{uid}/savedLocations/{key}` | a saved place: place, name, icon, color, list, and date saved |
| `users/{uid}/savedRoutes/{agencyId:routeId}` | a saved line: line details, favorite status, alert setting and hours, and date saved |
| `users/{uid}/sentAlerts/{id}` | alerts already sent to this rider (backend only) |
| `_pathlyCallableRateLimits` | per-user rate-limit counters (backend only, expire after 24 hours) |

Transit data is **not** stored in Firestore. Schedules live as versioned GTFS files in Cloud Storage, and live data is fetched from the agencies on demand. The active trip and trip history are stored on the device, so a trip can resume after the app restarts. Saved MTA maps also stay on the device.

---
### **User Authentication and Security**

Riders sign up and sign in with email and password or Google through Firebase Authentication. On the web, Google sign-in uses a Firebase popup. On phones it uses `expo-auth-session` and needs each platform's client ID, so the button stays hidden until one is set, and Expo Go can't complete the sign-in. Riders can export their data, and deleting an account removes the rider's Firestore documents, their data on the device, and their Auth account.

- Every Cloud Function requires a signed-in user.
- Backend keys (Google Maps server key, Swiftly, MTA Bus Time) are stored in Firebase Secret Manager and never shipped in the app. Keys that must be in the app (Maps client and Places) are restricted by platform and API.
- Each rider has request limits, enforced with a Firestore transaction so that extra server instances can't multiply them.
- App Check is ready on the backend. It is enforced for `geocodeAddress` now, and will be turned on for the transit functions once the web and native builds send valid App Check tokens.
- Admin tools are only available to configured admin emails.

---
### **Testing and CI**

| Suite | Runner | Tests | Covers |
|---|---|---|---|
| Backend (`firebase/functions/test/`) | Node test runner | 189 in 23 files | GTFS parsing per agency using recorded feeds, trip planning, live trip checks, service alerts, fares, walking paths, stop and line search, security of callable functions |
| Mobile (`mobile/__tests__/`) | Jest + React Native Testing Library | 423 in 45 files | screens and components, hooks (active trip, location), navigation logic, data services |

Our manual test cases, their results, and the bugs they found are in `TestPlan.ipynb`.

**CI:** a GitHub Actions workflow runs on every pull request and push to `main` and `services`. It installs dependencies, lints, runs both test suites, type-checks, and builds the app.

**Testing challenges:**
- **Live data changes constantly.** We record real feeds as fixtures so tests can replay delays, alerts and missing trips the same way every time.
- **Every agency's feed is different.** Each agency has its own test suite so that fixing one doesn't break another.
- **Location is hard to simulate.** The admin tools can fake the time and the rider's location, and step through a trip's checkpoints, so we can demo en-route navigation without riding the train.
- **Multiple platforms.** Map code differs between native and web, so visual issues still need manual checks on devices and emulators.
- **iOS Safari.** Scrolling, the keyboard, and the browser toolbars behave differently on a real iPhone, and our automated tests passed while these bugs were present. We test these on a real iPhone.

**Not covered yet:** end-to-end tests on a real device, tests against the Firebase emulators, and native iOS/Android builds in CI.

---
### **Development Workflow**

We continued using Claude and Codex with **OpenSpec**: before building a feature, we write a change proposal, design and task list, and archive it once it's done. More than 20 changes have been completed this way, including Google Maps search, real trip planning, MTA bus and ferry support, precomputed GTFS indexes and admin debug tools. En-route navigation is nearly finished, with one demo task left.

Each member's development cycle:

- create feature branch
- write or update the OpenSpec change
- develop locally (web, Android emulator, iOS simulator or Expo Go)
- run lint, tests and type-check locally
- pull request
- CI checks pass
- code review (by at least one other member)
- merge into main

Deploying is manual: `npm run deploy:backend` for the backend and `npm run deploy:web` for the web app. Before each deploy, Firebase rebuilds the GTFS indexes and stops if they or the walking tiles are out of date.

---
### **Next Steps**
- Rerouting and background navigation
- An iOS development build, to test Google sign-in and Google Maps on real phones
- Finish the open cases in the test plan, including Chrome on Android and phone rotation
- Friends and widgets
- Turn on App Check enforcement for the transit functions
- Run the deploy checks and backend lint in CI; add end-to-end tests
- Release preparation from M2: email service, landing page, analytics, app store listings